# End-to-End Pipeline: Public Compliance Data Analysis

**Complete workflow**: Data Ingestion → Bronze → Silver → Gold → Analysis → Results

This notebook executes the entire data pipeline end-to-end, from raw data collection through final analytical outputs.

---

## Notebook Sections

1. **Setup & Configuration** — Environment, paths, storage mode
2. **Data Ingestion (Bronze)** — Collect from IBGE, Transparency Portal, CGU
3. **Data Processing (Silver)** — Normalize, clean, join datasets
4. **Feature Engineering (Gold)** — Create analysis-ready datasets
5. **Exploratory Data Analysis** — Distributions, correlations, outliers
6. **Statistical Analysis** — OLS regression, hypothesis tests
7. **Machine Learning** — ElasticNet, Random Forest predictions
8. **Clustering Analysis** — K-means segmentation, PCA visualization
9. **Results Export** — Save outputs, figures, summary tables

**Estimated Runtime**: 15-30 minutes (depending on storage mode and data volume)

**Storage Modes Supported**:
- `local-only`: Store data locally (no AWS required)
- `s3-only`: Store data in S3 (AWS required)
- `both`: Store in both locations (redundancy)

## ⚠️ Security Note

Before sharing this notebook:
1. **Clear all outputs**: Kernel → Restart Kernel and Clear All Outputs
2. **Replace placeholder values**: Update S3_BUCKET and AWS_PROFILE with your actual values
3. **Never commit credentials**: Ensure no API keys or passwords are in the code

---


---

# 1. Setup & Configuration

## 1.1 Packages

In [9]:
# Standard library
import os
import sys
import json
import subprocess
from pathlib import Path
from datetime import datetime
import warnings

# Data manipulation
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics & ML
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression, ElasticNet
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_squared_error, r2_score, silhouette_score

# Statsmodels for OLS with robust SE
import statsmodels.api as sm
import statsmodels.formula.api as smf

# Configure paths
REPO_ROOT = Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))

# Import project modules
from src.analysis.data_loader import GoldDataLoader
from src.processing.gold_transformer import GoldTransformer

print("✅ All packages imported successfully")

✅ All packages imported successfully


## 1.2 Reproducibility

In [10]:
# Fixed random seed for reproducibility
SEED = 42
np.random.seed(SEED)

# Suppress non-critical warnings
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)

# Visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

print(f"✅ Reproducibility configured: SEED={SEED}")

✅ Reproducibility configured: SEED=42


## 1.3 Storage Configuration

Choose your storage mode: `local-only`, `s3-only`, or `both`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# CONFIGURATION: Choose your storage mode and paths
# ═══════════════════════════════════════════════════════════════════════════════

# Storage mode: 'local-only', 's3-only', or 'both'
STORAGE_MODE = 'local-only'  # Change to 's3-only' or 'both' as needed

# Local storage path (for local-only or both modes)
LOCAL_DATA_DIR = REPO_ROOT / 'data'  # Change to your preferred path

# S3 configuration (for s3-only or both modes)
import json as _json
_rtcfg_path = os.path.join('..', 'config', 'runtime_config.json')
if os.path.exists(_rtcfg_path):
    with open(_rtcfg_path) as _f:
        _rtcfg = _json.load(_f)
else:
    _rtcfg = {}
S3_BUCKET = os.environ.get('S3_BUCKET_NAME', _rtcfg.get('aws', {}).get('s3_bucket_name', ''))  # From env var or runtime_config.json
S3_PREFIX = 'bronze/'
AWS_PROFILE = None  # Or 'your-profile'  # Set to None for default credentials

# Create local directories if needed
if STORAGE_MODE in ('local-only', 'both'):
    for layer in ['bronze', 'silver', 'gold']:
        (LOCAL_DATA_DIR / layer).mkdir(parents=True, exist_ok=True)
    print(f"✅ Local directories created at: {LOCAL_DATA_DIR}")

# Export environment variables for shell scripts
os.environ['STORAGE_MODE'] = STORAGE_MODE
os.environ['LOCAL_DATA_DIR'] = str(LOCAL_DATA_DIR)
os.environ['S3_BUCKET'] = S3_BUCKET
if AWS_PROFILE:
    os.environ['AWS_PROFILE'] = AWS_PROFILE

print(f"📦 Storage Mode: {STORAGE_MODE}")
print(f"📁 Local Data Directory: ./{LOCAL_DATA_DIR.name}")
print(f"☁️  S3 Bucket: {S3_BUCKET} (prefix: {S3_PREFIX})")

## 1.4 Pipeline Configuration

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════════
# PIPELINE OPTIONS: Enable/disable pipeline stages
# ═══════════════════════════════════════════════════════════════════════════════

# Set to False to skip stages (useful for re-running specific parts)
RUN_INGESTION = True      # Step 2: Data Ingestion (Bronze)
RUN_SILVER = True         # Step 3: Silver Transformation
RUN_GOLD = True           # Step 4: Gold Feature Engineering
RUN_EDA = True            # Step 5: Exploratory Analysis
RUN_STATISTICS = True     # Step 6: Statistical Analysis
RUN_ML = True             # Step 7: Machine Learning
RUN_CLUSTERING = True     # Step 8: Clustering Analysis
RUN_EXPORT = True         # Step 9: Results Export

# Data scope
STATES_OF_INTEREST = None  # None for all 27 states, or list like ['SP', 'RJ', 'MG']
YEARS_OF_INTEREST = [2010, 2022]  # Census years to include

print("📋 Pipeline Configuration:")
print(f"   Ingestion: {RUN_INGESTION}")
print(f"   Silver: {RUN_SILVER}")
print(f"   Gold: {RUN_GOLD}")
print(f"   EDA: {RUN_EDA}")
print(f"   Statistics: {RUN_STATISTICS}")
print(f"   ML: {RUN_ML}")
print(f"   Clustering: {RUN_CLUSTERING}")
print(f"   Export: {RUN_EXPORT}")

---

# 2. Data Ingestion (Bronze Layer)

**Purpose**: Fetch raw data from external APIs and store with minimal transformation.

**Data Sources**:
- IBGE SIDRA API: Census data (population, income, literacy, sanitation)
- Transparency Portal: Federal transfers and compliance sanctions
- BCB API: IPCA inflation series for real income adjustment

**Output**: Bronze layer data (raw, source-aligned)

In [ ]:
if RUN_INGESTION:
    print("═" * 80)
    print("STEP 2: DATA INGESTION (Bronze Layer)")
    print("═" * 80)
    print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Storage Mode: {STORAGE_MODE}\n")
    
    # Run the Bronze ingestion script
    script_path = REPO_ROOT / 'scripts' / '01_bronze_ingestion.sh'
    
    cmd = [str(script_path)]
    if STORAGE_MODE == 'local-only':
        cmd.append('--local-only')
    
    print("Running Bronze ingestion...")
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_ROOT)
    
    if result.returncode == 0:
        print("✅ Bronze ingestion completed successfully")
    else:
        print("❌ Bronze ingestion failed")
        print("STDOUT:", result.stdout[-500:] if len(result.stdout) > 500 else result.stdout)
        print("STDERR:", result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    
    print(f"\nCompleted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("⏭️  Skipping Data Ingestion (RUN_INGESTION=False)")

### Verify Bronze Data

In [ ]:
if RUN_INGESTION or True:  # Always verify
    print("\n📁 Bronze Data Verification:")
    
    if STORAGE_MODE in ('local-only', 'both'):
        bronze_dir = LOCAL_DATA_DIR / 'bronze'
        if bronze_dir.exists():
            for source_dir in bronze_dir.iterdir():
                if source_dir.is_dir():
                    files = list(source_dir.glob('*'))
                    print(f"   {source_dir.name}: {len(files)} files")
        else:
            print(f"   ⚠️  Bronze directory not found: {bronze_dir}")
    
    if STORAGE_MODE in ('s3-only', 'both'):
        print(f"   ☁️  S3 bronze/ prefix: Check via AWS CLI: aws s3 ls s3://{S3_BUCKET}/bronze/")

---

# 3. Data Processing (Silver Layer)

**Purpose**: Normalize, clean, and join Bronze datasets into unified Silver tables.

**Transformations**:
- Schema alignment (consistent column names, types)
- Municipality code standardization (7-digit IBGE codes)
- Handle missing values and duplicates
- Join across data sources (census + transfers + sanctions)

**Output**: Silver layer tables (normalized, ready for analysis)

In [ ]:
if RUN_SILVER:
    print("\n═" * 80)
    print("STEP 3: SILVER TRANSFORMATION")
    print("═" * 80)
    print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    # Run the Silver transformation script
    script_path = REPO_ROOT / 'scripts' / '02_silver_transformation.sh'
    
    cmd = [str(script_path)]
    
    print("Running Silver transformation...")
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_ROOT)
    
    if result.returncode == 0:
        print("✅ Silver transformation completed successfully")
    else:
        print("❌ Silver transformation failed")
        print("STDERR:", result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    
    print(f"\nCompleted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("⏭️  Skipping Silver Transformation (RUN_SILVER=False)")

### Load Silver Data for Preview

In [ ]:
# Preview Silver layer data
silver_path = LOCAL_DATA_DIR / 'silver' if STORAGE_MODE in ('local-only', 'both') else None

if silver_path and silver_path.exists():
    silver_files = list(silver_path.glob('*.csv')) + list(silver_path.glob('*.parquet'))
    
    print(f"\n📁 Silver Layer Files ({len(silver_files)} files):")
    for f in silver_files[:5]:  # Show first 5
        print(f"   - {f.name}")
    if len(silver_files) > 5:
        print(f"   ... and {len(silver_files) - 5} more")
    
    # Load and preview one dataset
    if silver_files:
        sample_df = pd.read_csv(silver_files[0]) if silver_files[0].suffix == '.csv' else pd.read_parquet(silver_files[0])
        print(f"\n📊 Sample Silver Dataset: {silver_files[0].name}")
        print(f"   Shape: {sample_df.shape}")
        print(f"   Columns: {list(sample_df.columns[:5])}...")
        display(sample_df.head(3))

---

# 4. Feature Engineering (Gold Layer)

**Purpose**: Create analysis-ready datasets with derived features.

**Features Created**:
- Sanctions per 100k population (normalized metric)
- Real income (IPCA-adjusted to 2022 BRL)
- Regional indicators (Norte, Nordeste, etc.)
- Clustering features (PCA-ready)
- Municipality-year grain master table

**Output**: Gold layer datasets (analysis-ready)

In [ ]:
if RUN_GOLD:
    print("\n═" * 80)
    print("STEP 4: GOLD FEATURE ENGINEERING")
    print("═" * 80)
    print(f"\nStarted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    
    # Run the Gold transformation script
    script_path = REPO_ROOT / 'scripts' / '03_gold_transformation.sh'
    
    cmd = [str(script_path)]
    
    print("Running Gold feature engineering...")
    result = subprocess.run(cmd, capture_output=True, text=True, cwd=REPO_ROOT)
    
    if result.returncode == 0:
        print("✅ Gold transformation completed successfully")
    else:
        print("❌ Gold transformation failed")
        print("STDERR:", result.stderr[-500:] if len(result.stderr) > 500 else result.stderr)
    
    print(f"\nCompleted at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
else:
    print("⏭️  Skipping Gold Transformation (RUN_GOLD=False)")

### Load Gold Data for Analysis

In [ ]:
print("\n📊 Loading Gold Datasets for Analysis...")

# Use the GoldDataLoader
if STORAGE_MODE == 's3-only':
    loader = GoldDataLoader(
        bucket=S3_BUCKET,
        aws_profile=AWS_PROFILE if AWS_PROFILE else None,
        local_base_path=None
    )
else:
    # For local-only or both, prefer local for faster loading
    loader = GoldDataLoader(
        bucket=S3_BUCKET if STORAGE_MODE == 'both' else None,
        aws_profile=AWS_PROFILE if AWS_PROFILE else None,
        local_base_path=str(LOCAL_DATA_DIR / 'gold')
    )

# Load datasets
try:
    datasets = loader.load_all()
    
    print("\n✅ Gold datasets loaded successfully:")
    for name, df in datasets.items():
        print(f"   {name}: {df.shape}")
    
    # Assign to variables for easy access
    df_analysis = datasets.get('analysis_compliance')
    df_municipality = datasets.get('municipality_socioeconomic')
    df_state = datasets.get('state_summary')
    df_sanctions = datasets.get('sanctions_summary')
    df_clustering = datasets.get('consolidated_clustering')
    
except Exception as e:
    print(f"❌ Error loading gold datasets: {e}")
    print("⚠️  Attempting to load from alternative paths...")
    
    # Fallback: try loading directly from paths
    gold_path = LOCAL_DATA_DIR / 'gold'
    if gold_path.exists():
        csv_files = list(gold_path.glob('*.csv'))
        for f in csv_files:
            var_name = f.stem.replace('-', '_').replace(' ', '_')
            globals()[f'df_{var_name}'] = pd.read_csv(f)
            print(f"   Loaded {f.name} as df_{var_name}")

---

# 5. Exploratory Data Analysis

**Purpose**: Profile data quality, inspect distributions, identify patterns.

**Sections**:
- 5.1 Data Overview
- 5.2 Distribution Analysis
- 5.3 Regional Comparisons
- 5.4 Correlation Matrix

## 5.1 Data Overview

In [ ]:
if RUN_EDA and 'df_analysis' in globals() and df_analysis is not None:
    print("═" * 80)
    print("STEP 5: EXPLORATORY DATA ANALYSIS")
    print("═" * 80)
    
    print("\n📊 Primary Analysis Dataset Overview:")
    print(f"   Shape: {df_analysis.shape}")
    print(f"\n   Columns:")
    for col in df_analysis.columns:
        print(f"      - {col}")
    
    print(f"\n   Missing Values:")
    missing = df_analysis.isnull().sum()
    for col, count in missing[missing > 0].items():
        pct = count / len(df_analysis) * 100
        print(f"      {col}: {count} ({pct:.1f}%)")
    if missing.sum() == 0:
        print("      ✅ No missing values")
    
    print(f"\n   Data Types:")
    print(df_analysis.dtypes)
    
    display(df_analysis.describe())
else:
    print("⏭️  Skipping EDA (RUN_EDA=False or data not loaded)")

## 5.2 Distribution Analysis

In [ ]:
if RUN_EDA and 'df_analysis' in globals() and df_analysis is not None:
    
    # Key variables for visualization
    key_vars = ['sanctions_per_100k', 'income_avg', 'illiteracy_rate', 'urbanization_rate']
    key_vars = [v for v in key_vars if v in df_analysis.columns]
    
    if key_vars:
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        axes = axes.flatten()
        
        for i, var in enumerate(key_vars[:4]):
            if var in df_analysis.columns:
                axes[i].hist(df_analysis[var].dropna(), bins=30, edgecolor='black', alpha=0.7)
                axes[i].set_title(f'Distribution: {var}')
                axes[i].set_xlabel(var)
                axes[i].set_ylabel('Frequency')
        
        plt.suptitle('Key Variable Distributions', fontsize=14, y=1.02)
        plt.tight_layout()
        plt.show()
        
        # Save figure
        fig.savefig(REPO_ROOT / 'outputs' / 'eda_distributions.png', dpi=300, bbox_inches='tight')
        print("\n✅ Distribution plots saved to outputs/eda_distributions.png")

## 5.3 Regional Comparisons

In [ ]:
if RUN_EDA and 'df_analysis' in globals() and df_analysis is not None:
    
    if 'region' in df_analysis.columns and 'sanctions_per_100k' in df_analysis.columns:
        
        fig, ax = plt.subplots(figsize=(12, 6))
        
        regional_data = df_analysis.groupby('region')['sanctions_per_100k'].mean().sort_values(ascending=False)
        
        regional_data.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
        ax.set_title('Average Sanctions per 100k Population by Region', fontsize=12)
        ax.set_xlabel('Region')
        ax.set_ylabel('Sanctions per 100k')
        ax.tick_params(axis='x', rotation=45)
        
        plt.tight_layout()
        plt.show()
        
        # Save figure
        fig.savefig(REPO_ROOT / 'outputs' / 'regional_sanctions.png', dpi=300, bbox_inches='tight')
        
        print("\n📊 Regional Sanctions Summary:")
        print(regional_data.to_frame().round(2))

## 5.4 Correlation Matrix

In [ ]:
if RUN_EDA and 'df_analysis' in globals() and df_analysis is not None:
    
    # Select numeric columns for correlation
    numeric_cols = df_analysis.select_dtypes(include=[np.number]).columns
    numeric_cols = [c for c in numeric_cols if not c.endswith('_id') and c != 'year']
    
    if len(numeric_cols) > 2:
        
        # Calculate correlation matrix
        corr_matrix = df_analysis[numeric_cols].corr()
        
        # Plot
        fig, ax = plt.subplots(figsize=(12, 10))
        sns.heatmap(corr_matrix, annot=True, cmap='RdBu_r', center=0, 
                    square=True, fmt='.2f', cbar_kws={"shrink": .8}, ax=ax)
        ax.set_title('Correlation Matrix of Key Variables', fontsize=12)
        
        plt.tight_layout()
        plt.show()
        
        # Save figure
        fig.savefig(REPO_ROOT / 'outputs' / 'correlation_matrix.png', dpi=300, bbox_inches='tight')
        
        # Print strongest correlations with sanctions
        if 'sanctions_per_100k' in corr_matrix.columns:
            sanctions_corr = corr_matrix['sanctions_per_100k'].drop('sanctions_per_100k').abs().sort_values(ascending=False)
            print("\n📊 Strongest Correlations with Sanctions per 100k:")
            for var, corr in sanctions_corr.head(5).items():
                direction = "positive" if corr_matrix['sanctions_per_100k'][var] > 0 else "negative"
                print(f"   {var}: r = {corr_matrix['sanctions_per_100k'][var]:.3f} ({direction})")

---

# 6. Statistical Analysis

**Purpose**: OLS regression modeling with robust standard errors.

**Models**:
- Model 1: Income only (baseline)
- Model 2: Income + controls (literacy, urbanization)
- Model 3: Full model with regional effects

In [ ]:
if RUN_STATISTICS and 'df_analysis' in globals() and df_analysis is not None:
    
    print("\n═" * 80)
    print("STEP 6: STATISTICAL ANALYSIS (OLS Regression)")
    print("═" * 80)
    
    # Prepare data
    model_df = df_analysis.copy()
    
    # Define variables
    dependent = 'sanctions_per_100k'
    independent = [
        'income_avg',
        'illiteracy_rate',
        'urbanization_rate',
        'gini_index'
    ]
    
    # Filter to available columns
    available_vars = [v for v in [dependent] + independent if v in model_df.columns]
    
    if len(available_vars) < 2:
        print("⚠️  Insufficient variables for regression. Available columns:")
        print(list(model_df.columns))
    else:
        # Create complete cases
        regression_df = model_df[available_vars].dropna()
        
        print(f"\n📊 Regression Dataset: {regression_df.shape[0]} observations, {len(available_vars)} variables")
        
        # Model 1: Income only
        if 'income_avg' in regression_df.columns:
            print("\n📈 Model 1: Income Only")
            X1 = sm.add_constant(regression_df['income_avg'])
            y = regression_df[dependent]
            model1 = sm.OLS(y, X1).fit(cov_type='HC3')  # Robust SE
            print(f"   R² = {model1.rsquared:.3f}")
            print(f"   Income coefficient: {model1.params['income_avg']:.4f} (p={model1.pvalues['income_avg']:.4f})")
        
        # Model 2: Full specification
        independent_available = [v for v in independent if v in regression_df.columns]
        if len(independent_available) >= 2:
            print("\n📈 Model 2: Full Specification")
            X2 = sm.add_constant(regression_df[independent_available])
            model2 = sm.OLS(y, X2).fit(cov_type='HC3')
            print(f"   R² = {model2.rsquared:.3f}")
            print(f"   Adjusted R² = {model2.rsquared_adj:.3f}")
            print("\n   Coefficients:")
            for var in independent_available:
                coef = model2.params[var]
                pval = model2.pvalues[var]
                stars = "***" if pval < 0.001 else "**" if pval < 0.01 else "*" if pval < 0.05 else ""
                print(f"      {var}: {coef:.4f} (p={pval:.4f}){stars}")
        
        # Store results for summary
        regression_results = {
            'model1_r2': model1.rsquared if 'model1' in locals() else None,
            'model2_r2': model2.rsquared if 'model2' in locals() else None,
            'income_coef': model2.params['income_avg'] if 'model2' in locals() and 'income_avg' in model2.params else None
        }
else:
    print("⏭️  Skipping Statistical Analysis (RUN_STATISTICS=False or data not loaded)")

---

# 7. Machine Learning

**Purpose**: Supervised learning models for sanctions prediction.

**Models**:
- ElasticNet (regularized linear)
- Random Forest (ensemble, non-linear)

In [ ]:
if RUN_ML and 'df_analysis' in globals() and df_analysis is not None:
    
    print("\n═" * 80)
    print("STEP 7: MACHINE LEARNING")
    print("═" * 80)
    
    # Prepare ML dataset
    ml_vars = ['sanctions_per_100k', 'income_avg', 'illiteracy_rate', 'urbanization_rate']
    ml_vars = [v for v in ml_vars if v in df_analysis.columns]
    
    if len(ml_vars) < 2:
        print("⚠️  Insufficient variables for ML")
    else:
        ml_df = df_analysis[ml_vars].dropna()
        
        X = ml_df.drop('sanctions_per_100k', axis=1)
        y = ml_df['sanctions_per_100k']
        
        # Train/test split
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=0.2, random_state=SEED
        )
        
        print(f"\n📊 ML Dataset: {len(X_train)} train, {len(X_test)} test samples")
        print(f"   Features: {list(X.columns)}")
        
        # Scale features
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Model 1: ElasticNet
        print("\n📈 Model 1: ElasticNet")
        enet = ElasticNet(alpha=0.1, l1_ratio=0.5, random_state=SEED)
        enet.fit(X_train_scaled, y_train)
        enet_pred = enet.predict(X_test_scaled)
        enet_r2 = r2_score(y_test, enet_pred)
        enet_rmse = np.sqrt(mean_squared_error(y_test, enet_pred))
        
        print(f"   R² (test): {enet_r2:.3f}")
        print(f"   RMSE: {enet_rmse:.2f}")
        print(f"   Non-zero coefficients: {np.sum(enet.coef_ != 0)}/{len(enet.coef_)}")
        
        # Model 2: Random Forest
        print("\n📈 Model 2: Random Forest")
        rf = RandomForestRegressor(
            n_estimators=100,
            max_depth=10,
            random_state=SEED,
            n_jobs=-1
        )
        rf.fit(X_train, y_train)  # RF doesn't require scaling
        rf_pred = rf.predict(X_test)
        rf_r2 = r2_score(y_test, rf_pred)
        rf_rmse = np.sqrt(mean_squared_error(y_test, rf_pred))
        
        print(f"   R² (test): {rf_r2:.3f}")
        print(f"   RMSE: {rf_rmse:.2f}")
        
        # Feature importance
        print("\n📊 Feature Importance (Random Forest):")
        importance = pd.Series(rf.feature_importances_, index=X.columns)
        importance = importance.sort_values(ascending=False)
        for feat, imp in importance.items():
            print(f"   {feat}: {imp:.3f}")
        
        # Model comparison plot
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))
        
        # ElasticNet predictions
        axes[0].scatter(y_test, enet_pred, alpha=0.6, edgecolors='black', linewidth=0.5)
        axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        axes[0].set_xlabel('Actual')
        axes[0].set_ylabel('Predicted')
        axes[0].set_title(f'ElasticNet (R² = {enet_r2:.3f})')
        
        # Random Forest predictions
        axes[1].scatter(y_test, rf_pred, alpha=0.6, edgecolors='black', linewidth=0.5, color='green')
        axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        axes[1].set_xlabel('Actual')
        axes[1].set_ylabel('Predicted')
        axes[1].set_title(f'Random Forest (R² = {rf_r2:.3f})')
        
        plt.suptitle('Machine Learning Model Performance', fontsize=14)
        plt.tight_layout()
        plt.show()
        
        # Save figure
        fig.savefig(REPO_ROOT / 'outputs' / 'ml_predictions.png', dpi=300, bbox_inches='tight')
        
        # Store results
        ml_results = {
            'elasticnet_r2': enet_r2,
            'elasticnet_rmse': enet_rmse,
            'rf_r2': rf_r2,
            'rf_rmse': rf_rmse,
            'best_model': 'Random Forest' if rf_r2 > enet_r2 else 'ElasticNet'
        }
else:
    print("⏭️  Skipping Machine Learning (RUN_ML=False or data not loaded)")

---

# 8. Clustering Analysis

**Purpose**: Unsupervised segmentation of municipalities.

**Method**: K-means clustering with PCA visualization

In [ ]:
if RUN_CLUSTERING and 'df_clustering' in globals() and df_clustering is not None:
    
    print("\n═" * 80)
    print("STEP 8: CLUSTERING ANALYSIS")
    print("═" * 80)
    
    # Prepare clustering data
    cluster_vars = ['income_avg', 'illiteracy_rate', 'urbanization_rate', 'sanctions_per_100k']
    cluster_vars = [v for v in cluster_vars if v in df_clustering.columns]
    
    if len(cluster_vars) < 2:
        print("⚠️  Insufficient variables for clustering")
    else:
        cluster_df = df_clustering[cluster_vars].dropna()
        
        print(f"\n📊 Clustering Dataset: {cluster_df.shape[0]} municipalities, {len(cluster_vars)} features")
        
        # Scale features
        scaler = StandardScaler()
        cluster_scaled = scaler.fit_transform(cluster_df)
        
        # Determine optimal K (elbow method simplified)
        K = 4  # Based on prior analysis: 2 bulk + 2 outlier
        
        # Fit K-means
        print(f"\n📈 K-Means Clustering (K={K})")
        kmeans = KMeans(n_clusters=K, random_state=SEED, n_init=10)
        cluster_labels = kmeans.fit_predict(cluster_scaled)
        
        # Silhouette score
        sil_score = silhouette_score(cluster_scaled, cluster_labels)
        print(f"   Silhouette Score: {sil_score:.3f}")
        
        # Cluster sizes
        cluster_sizes = pd.Series(cluster_labels).value_counts().sort_index()
        print(f"\n   Cluster Sizes:")
        for i, size in cluster_sizes.items():
            print(f"      Cluster {i}: {size} municipalities")
        
        # Cluster characteristics
        cluster_df['cluster'] = cluster_labels
        print(f"\n📊 Cluster Characteristics (mean values):")
        cluster_summary = cluster_df.groupby('cluster')[cluster_vars].mean()
        print(cluster_summary.round(2))
        
        # PCA for visualization
        print("\n📈 PCA Visualization")
        pca = PCA(n_components=2)
        pca_result = pca.fit_transform(cluster_scaled)
        
        print(f"   Explained Variance: PC1={pca.explained_variance_ratio_[0]:.1%}, PC2={pca.explained_variance_ratio_[1]:.1%}")
        print(f"   Total: {sum(pca.explained_variance_ratio_):.1%}")
        
        # Plot
        fig, ax = plt.subplots(figsize=(10, 8))
        
        colors = ['blue', 'green', 'orange', 'red']
        for i in range(K):
            mask = cluster_labels == i
            ax.scatter(
                pca_result[mask, 0],
                pca_result[mask, 1],
                c=colors[i],
                label=f'Cluster {i} (n={cluster_sizes[i]})',
                alpha=0.6,
                edgecolors='black',
                linewidth=0.5
            )
        
        ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
        ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
        ax.set_title('Municipality Clusters (K-Means + PCA)')
        ax.legend()
        ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.show()
        
        # Save figure
        fig.savefig(REPO_ROOT / 'outputs' / 'clustering_pca.png', dpi=300, bbox_inches='tight')
        
        # Store results
        clustering_results = {
            'k': K,
            'silhouette_score': sil_score,
            'cluster_sizes': cluster_sizes.to_dict(),
            'pca_variance': sum(pca.explained_variance_ratio_)
        }
else:
    print("⏭️  Skipping Clustering (RUN_CLUSTERING=False or data not loaded)")

---

# 9. Results Export

**Purpose**: Save outputs, figures, and summary tables for thesis integration.

In [ ]:
if RUN_EXPORT:
    
    print("\n═" * 80)
    print("STEP 9: RESULTS EXPORT")
    print("═" * 80)
    
    # Create outputs directory
    outputs_dir = REPO_ROOT / 'outputs'
    outputs_dir.mkdir(exist_ok=True)
    
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    
    # Export 1: Summary statistics
    summary_data = {
        'run_timestamp': timestamp,
        'storage_mode': STORAGE_MODE,
        'data_sources': ['IBGE', 'Transparency Portal', 'CGU Sanctions'],
    }
    
    # Add regression results if available
    if 'regression_results' in globals():
        summary_data['regression'] = regression_results
    
    # Add ML results if available
    if 'ml_results' in globals():
        summary_data['machine_learning'] = ml_results
    
    # Add clustering results if available
    if 'clustering_results' in globals():
        summary_data['clustering'] = clustering_results
    
    # Save summary JSON
    summary_path = outputs_dir / f'analysis_summary_{timestamp}.json'
    with open(summary_path, 'w') as f:
        json.dump(summary_data, f, indent=2, default=str)
    print(f"\n✅ Summary saved: {summary_path}")
    
    # Export 2: Data tables (if analysis data available)
    if 'df_analysis' in globals() and df_analysis is not None:
        # State summary
        if 'state_code' in df_analysis.columns:
            state_summary = df_analysis.groupby('state_code').agg({
                'sanctions_per_100k': 'mean',
                'income_avg': 'mean'
            }).round(2)
            state_summary.to_csv(outputs_dir / 'state_summary.csv')
            print("✅ State summary exported")
    
    # Export 3: Key figures list
    figures = list(outputs_dir.glob('*.png'))
    print(f"\n📊 Generated Figures ({len(figures)}):")
    for fig in sorted(figures):
        print(f"   - {fig.name}")
    
    print(f"\n✅ All outputs saved to: {outputs_dir}")
    
    # Final summary
    print("\n" + "═" * 80)
    print("PIPELINE COMPLETE")
    print("═" * 80)
    print(f"\nTimestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"Storage Mode: {STORAGE_MODE}")
    print(f"Output Directory: {outputs_dir}")
    
else:
    print("⏭️  Skipping Results Export (RUN_EXPORT=False)")

---

# Appendix: Quick Reference

## Running Specific Sections

To re-run only specific parts, change the configuration in Section 1.4:

```python
# Example: Run only analysis (skip ingestion/processing)
RUN_INGESTION = False
RUN_SILVER = False
RUN_GOLD = False
RUN_EDA = True
RUN_STATISTICS = True
RUN_ML = True
RUN_CLUSTERING = True
```

## Storage Mode Quick Switch

```python
# Local-only (no AWS)
STORAGE_MODE = 'local-only'
LOCAL_DATA_DIR = Path('/path/to/data')

# S3-only (AWS required)
STORAGE_MODE = 's3-only'
S3_BUCKET = 'your-bucket'

# Both (redundancy)
STORAGE_MODE = 'both'
```

## Expected Runtime

| Stage | Local | S3 | Both |
|-------|-------|-----|------|
| Ingestion | 5-10 min | 5-10 min | 10-15 min |
| Silver | 2-3 min | 2-3 min | 3-5 min |
| Gold | 1-2 min | 1-2 min | 2-3 min |
| Analysis | 2-3 min | 2-3 min | 2-3 min |
| **Total** | **10-18 min** | **10-18 min** | **17-26 min** |

## Output Files

The pipeline generates:
- `outputs/eda_distributions.png` — Variable distributions
- `outputs/regional_sanctions.png` — Regional comparison
- `outputs/correlation_matrix.png` — Correlation heatmap
- `outputs/ml_predictions.png` — ML model performance
- `outputs/clustering_pca.png` — Cluster visualization
- `outputs/analysis_summary_YYYYMMDD_HHMMSS.json` — Results summary
- `outputs/state_summary.csv` — State-level statistics

---

## Troubleshooting

### Missing data files
```python
# Re-run specific pipeline stage
RUN_INGESTION = True
RUN_SILVER = True
RUN_GOLD = True
```

### AWS authentication errors
```bash
# Verify credentials
aws sts get-caller-identity

# Or switch to local-only mode
STORAGE_MODE = 'local-only'
```

### Memory issues with large datasets
```python
# Filter to specific states
STATES_OF_INTEREST = ['SP', 'RJ', 'MG']
```